### Фінальний проект

Студент: Євген Галенко

Група: AI&ML 4

In [4]:
!pip install -q pgserver sqlalchemy psycopg2-binary pandas sqlparse duckdb gdown


In [5]:
import pandas as pd
import pgserver
import sqlparse
from sqlalchemy import create_engine, text

pg = pgserver.get_server("/tmp/hw_fp_pg")
engine = create_engine(pg.get_uri())

def run_sql(sql: str):
    with engine.begin() as conn:
        for statement in sqlparse.split(sql):
            statement = statement.strip()
            if not statement:
                continue
            result = conn.execute(text(statement))
            if result.returns_rows:
                display(pd.DataFrame(result.fetchall(), columns=result.keys()))
            else:
                print(f"OK: {result.rowcount} rows")

In [6]:
from IPython.display import display, Markdown
run_sql("SELECT version() AS postgresql_version;")

,postgresql_version
0,"PostgreSQL 16.2 on x86_64-pc-linux-gnu, compil..."


In [8]:
# 1. Створення схеми, завантаження CSV і перегляд даних
# 1.1. Створення pandemic і встановлення search_path
run_sql("""
    CREATE SCHEMA IF NOT EXISTS pandemic;
    SET search_path TO pandemic, public;
    SELECT current_schema() AS active_schema;
""")

OK: -1 rows
OK: -1 rows


,active_schema
0,pandemic


In [9]:
# 1.2. Завантаження офіційного CSV за посиланням
from pathlib import Path
import gdown

csv_path = Path("/content/infectious_cases.csv")
file_id = "1dQf-uVXzYPYYGgXsG1y4Q1qDdhP2miZl"

try:
    downloaded = gdown.download(
        id=file_id,
        output=str(csv_path),
        quiet=False,
    )
except Exception as exc:
    downloaded = None
    print(f"Автоматичне завантаження недоступне: {exc}")

if not downloaded or not csv_path.is_file() or csv_path.stat().st_size == 0:
    print("Завантажте infectious_cases.csv вручну з посилання в умові.")
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("Не вдалося отримати infectious_cases.csv")
    source_name = next(iter(uploaded))
    Path(source_name).replace(csv_path)

print(f"CSV: {csv_path} | {csv_path.stat().st_size:,} bytes")

Downloading...
From: https://drive.google.com/uc?id=1dQf-uVXzYPYYGgXsG1y4Q1qDdhP2miZl
To: /content/infectious_cases.csv
100%|██████████| 541k/541k [00:00<00:00, 8.96MB/s]

CSV: /content/infectious_cases.csv | 540,895 bytes


In [11]:
# 1.3. Огляд через DuckDB та перевірка в pandas

# Я раніше ніколи не користувався DuckDB тому було цікаво поексперементувати з ним
import duckdb
from IPython.display import display, Markdown

con = duckdb.connect(database=":memory:")

csv_summary = con.execute("""
    SELECT COUNT(*) AS source_rows
    FROM read_csv(?, header=true, all_varchar=true)
""", [str(csv_path)]).df()

display(csv_summary)
display(con.execute("""
    SELECT *
    FROM read_csv(?, header=true, all_varchar=true)
    LIMIT 5
""", [str(csv_path)]).df())

,source_rows
0,10521


,Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,Afghanistan,AFG,1980,None,6160,0,None,None,None,None,None,None
1,Afghanistan,AFG,1981,None,5859,0,None,None,None,None,None,None
2,Afghanistan,AFG,1982,None,9730,0,None,None,None,None,None,None
3,Afghanistan,AFG,1983,None,13937,0,None,None,None,None,None,None
4,Afghanistan,AFG,1984,None,3864,0,None,None,None,None,None,None


In [12]:
import re

# Для імпорту використовуємо текстовий staging: жодних прихованих приведень типів.
raw_df = pd.read_csv(
    csv_path,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig",
    low_memory=False,
)
raw_df.columns = [str(column).strip().lstrip("\ufeff") for column in raw_df.columns]

# Оригінальні заголовки стають SQL-ідентифікаторами — перевіряємо їх.
assert raw_df.columns.is_unique, "CSV містить дублікати назв колонок"
assert all(re.fullmatch(r"[A-Za-z_][A-Za-z_0-9]*", c) for c in raw_df.columns), (
    "Неочікувані назви колонок у CSV; перевірте заголовки файлу"
)

# Приводимо лише обов'язкові заголовки до назв, заданих у завданні.
required_columns = ["Entity", "Code", "Year", "Number_rabies"]
column_lookup = {col.lower(): col for col in raw_df.columns}
missing = [col for col in required_columns if col.lower() not in column_lookup]
if missing:
    raise ValueError(f"У CSV бракує обов'язкових колонок: {missing}")
raw_df = raw_df.rename(columns={column_lookup[c.lower()]: c for c in required_columns})

assert raw_df["Entity"].str.strip().ne("").all(), "Порожні значення Entity"
assert raw_df["Year"].str.strip().str.fullmatch(r"[0-9]{1,4}").all(), (
    "Некоректні значення Year (очікуються цілі роки)"
)

columns = list(raw_df.columns)
metric_columns = [col for col in columns if col not in {"Entity", "Code", "Year"}]
assert "Number_rabies" in metric_columns

print(f"Рядків: {len(raw_df):,} | колонок: {len(columns)}")
print("Колонки:", columns)
display(raw_df.head())

Рядків: 10,521 | колонок: 12
Колонки: ['Entity', 'Code', 'Year', 'Number_yaws', 'polio_cases', 'cases_guinea_worm', 'Number_rabies', 'Number_malaria', 'Number_hiv', 'Number_tuberculosis', 'Number_smallpox', 'Number_cholera_cases']


,Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,Afghanistan,AFG,1980,,6160,0,,,,,,
1,Afghanistan,AFG,1981,,5859,0,,,,,,
2,Afghanistan,AFG,1982,,9730,0,,,,,,
3,Afghanistan,AFG,1983,,13937,0,,,,,,
4,Afghanistan,AFG,1984,,3864,0,,,,,,


In [13]:
# 1.4. Імпорт сирих даних
def q(identifier: str) -> str:
    """Безпечне SQL-квотування імені колонки з перевіреного CSV."""
    return '"' + identifier.replace('"', '""') + '"'

staging_column_definitions = ",\n    ".join(f"{q(col)} TEXT" for col in columns)

run_sql(f"""
    DROP TABLE IF EXISTS pandemic.disease_statistics;
    DROP TABLE IF EXISTS pandemic.entities;
    DROP TABLE IF EXISTS pandemic.infectious_cases;

    CREATE TABLE pandemic.infectious_cases (
        {staging_column_definitions}
    );
""")

raw_df.to_sql(
    name="infectious_cases",
    con=engine,
    schema="pandemic",
    if_exists="append",
    index=False,
    chunksize=1000,
    method="multi",
)

run_sql("""
    SELECT COUNT(*) AS imported_rows
    FROM pandemic.infectious_cases;

    SELECT *
    FROM pandemic.infectious_cases
    LIMIT 5;
""")

OK: -1 rows
OK: -1 rows
OK: -1 rows
OK: -1 rows


,imported_rows
0,10521


,Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,Afghanistan,AFG,1980,,6160,0,,,,,,
1,Afghanistan,AFG,1981,,5859,0,,,,,,
2,Afghanistan,AFG,1982,,9730,0,,,,,,
3,Afghanistan,AFG,1983,,13937,0,,,,,,
4,Afghanistan,AFG,1984,,3864,0,,,,,,


In [14]:
# 2. Нормалізація до 3НФ
metric_definitions = ",\n    ".join(
    f"{q(col)} NUMERIC" for col in metric_columns
)

run_sql(f"""
    CREATE TABLE pandemic.entities (
        id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        "Entity" TEXT NOT NULL CHECK ("Entity" <> ''),
        "Code" TEXT,
        CONSTRAINT uq_entities_entity_code
            UNIQUE NULLS NOT DISTINCT ("Entity", "Code")
    );

    CREATE TABLE pandemic.disease_statistics (
        id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        entity_id BIGINT NOT NULL REFERENCES pandemic.entities(id),
        "Year" INTEGER NOT NULL CHECK ("Year" BETWEEN 1 AND 9999),
        {metric_definitions}
    );

    CREATE INDEX ix_disease_statistics_entity_year
        ON pandemic.disease_statistics (entity_id, "Year");
""")

OK: -1 rows
OK: -1 rows
OK: -1 rows


In [15]:
# 2.2. Заповнення довідника унікальних комбінацій Entity, Code
run_sql("""
    INSERT INTO pandemic.entities ("Entity", "Code")
    SELECT DISTINCT
        BTRIM("Entity") AS "Entity",
        NULLIF(BTRIM("Code"), '') AS "Code"
    FROM pandemic.infectious_cases
    WHERE NULLIF(BTRIM("Entity"), '') IS NOT NULL
    ON CONFLICT ON CONSTRAINT uq_entities_entity_code DO NOTHING;

    SELECT COUNT(*) AS unique_entity_code_pairs
    FROM pandemic.entities;

    SELECT *
    FROM pandemic.entities
    ORDER BY id
    LIMIT 10;
""")

OK: 245 rows


,unique_entity_code_pairs
0,245


,id,Entity,Code
0,1,Russia,RUS
1,2,Trinidad and Tobago,TTO
2,3,Antigua and Barbuda,ATG
3,4,Nicaragua,NIC
4,5,Denmark,DNK
5,6,Turkey,TUR
6,7,Sudan,SDN
7,8,Myanmar,MMR
8,9,Macao,MAC
9,10,Togo,TGO


In [16]:
# 2.3. Перенесення всіх кількісних показників
# Простий числовий формат
number_pattern = r"^[+-]?([0-9]+(\.[0-9]*)?|\.[0-9]+)([eE][+-]?[0-9]+)?$"

numeric_selects = []
for column in metric_columns:
    normalized_value = f"REPLACE(BTRIM(src.{q(column)}), ',', '')"
    numeric_selects.append(f"""
        CASE
            WHEN {normalized_value} ~ '{number_pattern}'
            THEN {normalized_value}::NUMERIC
            ELSE NULL
        END AS {q(column)}
    """)

metrics_sql = ",\n".join(numeric_selects)
insert_columns = ", ".join(q(col) for col in metric_columns)

run_sql(f"""
    INSERT INTO pandemic.disease_statistics (
        entity_id, "Year", {insert_columns}
    )
    SELECT
        dim.id AS entity_id,
        BTRIM(src."Year")::INTEGER AS "Year",
        {metrics_sql}
    FROM pandemic.infectious_cases AS src
    JOIN pandemic.entities AS dim
        ON dim."Entity" = BTRIM(src."Entity")
       AND dim."Code" IS NOT DISTINCT FROM NULLIF(BTRIM(src."Code"), '');

    ANALYZE pandemic.entities;
    ANALYZE pandemic.disease_statistics;
""")

OK: 10521 rows
OK: -1 rows
OK: -1 rows


In [17]:
# 2.4. Перевірка нормалізації
validation_query = """
    SELECT
        (SELECT COUNT(*) FROM pandemic.infectious_cases) AS original_rows,
        (SELECT COUNT(*) FROM pandemic.entities) AS entities_count,
        (SELECT COUNT(*) FROM pandemic.disease_statistics) AS facts_count,
        (
            SELECT COUNT(*)
            FROM pandemic.disease_statistics AS fact
            LEFT JOIN pandemic.entities AS dim ON dim.id = fact.entity_id
            WHERE dim.id IS NULL
        ) AS orphan_facts;
"""

run_sql(validation_query)

with engine.connect() as conn:
    counts = conn.execute(text(validation_query)).mappings().one()

assert counts["original_rows"] == counts["facts_count"], (
    "Кількість рядків змінилася під час нормалізації"
)
assert counts["orphan_facts"] == 0, "Виявлено факти без відповідного Entity"
print("✅ Кількість рядків збережена; зовнішні ключі коректні.")

run_sql("""
    SELECT
        dim."Entity",
        dim."Code",
        fact."Year",
        fact."Number_rabies"
    FROM pandemic.disease_statistics AS fact
    JOIN pandemic.entities AS dim ON dim.id = fact.entity_id
    ORDER BY fact.id
    LIMIT 10;
""")

,original_rows,entities_count,facts_count,orphan_facts
0,10521,245,10521,0


✅ Кількість рядків збережена; зовнішні ключі коректні.


,Entity,Code,Year,Number_rabies
0,Afghanistan,AFG,1980,None
1,Afghanistan,AFG,1981,None
2,Afghanistan,AFG,1982,None
3,Afghanistan,AFG,1983,None
4,Afghanistan,AFG,1984,None
5,Afghanistan,AFG,1985,None
6,Afghanistan,AFG,1986,None
7,Afghanistan,AFG,1987,None
8,Afghanistan,AFG,1988,None
9,Afghanistan,AFG,1989,None


In [18]:
# 3. Агрегація Number_rabies для кожної пари
run_sql("""
    SELECT
        dim.id AS entity_id,
        dim."Entity",
        dim."Code",
        ROUND(AVG(fact."Number_rabies"), 2) AS avg_rabies,
        MIN(fact."Number_rabies") AS min_rabies,
        MAX(fact."Number_rabies") AS max_rabies,
        SUM(fact."Number_rabies") AS sum_rabies
    FROM pandemic.entities AS dim
    JOIN pandemic.disease_statistics AS fact
        ON fact.entity_id = dim.id
    GROUP BY dim.id, dim."Entity", dim."Code"
    ORDER BY avg_rabies DESC NULLS LAST, dim."Entity"
    LIMIT 10;
""")

,entity_id,Entity,Code,avg_rabies,min_rabies,max_rabies,sum_rabies
0,84,World,OWID_WRL,20192.37,14075.508,24744.658,605771.111
1,44,Lower Middle Income (WB),None,15193.96,10202.528,19182.795,455818.7815
2,158,South Asia (WB),None,11729.89,7271.2754,15361.878,351896.6626
3,139,South-East Asia Region (WHO),None,11424.33,6806.0127,15641.958,342729.8068
4,66,G20,None,10189.05,6339.0796,13164.881,305671.3801
5,223,India,IND,8599.17,5425.8726,11121.142,257975.1984
6,119,Sub-Saharan Africa (WB),None,4748.27,4356.8184,5017.977,142448.0427
7,77,African Region (WHO),None,4649.70,4283.073,4917.784,139491.0713
8,193,Low Income (WB),None,3568.11,3014.617,3873.533,107043.1635
9,178,East Asia & Pacific (WB),None,3427.46,2009.9395,4591.0107,102823.8340


In [19]:
# 4. Колонки дати та різниці в роках за вбудованими SQL-функціями
run_sql("""
    SELECT
        fact.id,
        dim."Entity",
        fact."Year",
        MAKE_DATE(fact."Year", 1, 1) AS first_january,
        CURRENT_DATE AS today,
        EXTRACT(
            YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(fact."Year", 1, 1))
        )::INTEGER AS years_difference
    FROM pandemic.disease_statistics AS fact
    JOIN pandemic.entities AS dim ON dim.id = fact.entity_id
    ORDER BY fact.id
    LIMIT 10;
""")

,id,Entity,Year,first_january,today,years_difference
0,1,Afghanistan,1980,1980-01-01,2026-11-07,46
1,2,Afghanistan,1981,1981-01-01,2026-11-07,45
2,3,Afghanistan,1982,1982-01-01,2026-11-07,44
3,4,Afghanistan,1983,1983-01-01,2026-11-07,43
4,5,Afghanistan,1984,1984-01-01,2026-11-07,42
5,6,Afghanistan,1985,1985-01-01,2026-11-07,41
6,7,Afghanistan,1986,1986-01-01,2026-11-07,40
7,8,Afghanistan,1987,1987-01-01,2026-11-07,39
8,9,Afghanistan,1988,1988-01-01,2026-11-07,38
9,10,Afghanistan,1989,1989-01-01,2026-11-07,37


In [20]:
# 5. Дві власні SQL-функції
# 5.1. Функція для обчислення різниці в роках
run_sql("""
    CREATE OR REPLACE FUNCTION pandemic.years_since(p_year INTEGER)
    RETURNS INTEGER
    LANGUAGE SQL
    STABLE
    AS $$
        SELECT EXTRACT(
            YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(p_year, 1, 1))
        )::INTEGER;
    $$;

    SELECT
        1996 AS input_year,
        MAKE_DATE(1996, 1, 1) AS first_january,
        CURRENT_DATE AS today,
        pandemic.years_since(1996) AS years_difference;
""")

OK: -1 rows


,input_year,first_january,today,years_difference
0,1996,1996-01-01,2026-11-07,30


In [21]:
run_sql("""
    SELECT
        fact."Year",
        MAKE_DATE(fact."Year", 1, 1) AS first_january,
        CURRENT_DATE AS today,
        pandemic.years_since(fact."Year") AS years_difference
    FROM pandemic.disease_statistics AS fact
    ORDER BY fact.id
    LIMIT 10;
""")

,Year,first_january,today,years_difference
0,1980,1980-01-01,2026-11-07,46
1,1981,1981-01-01,2026-11-07,45
2,1982,1982-01-01,2026-11-07,44
3,1983,1983-01-01,2026-11-07,43
4,1984,1984-01-01,2026-11-07,42
5,1985,1985-01-01,2026-11-07,41
6,1986,1986-01-01,2026-11-07,40
7,1987,1987-01-01,2026-11-07,39
8,1988,1988-01-01,2026-11-07,38
9,1989,1989-01-01,2026-11-07,37


In [22]:
# 5.2. Функція для обчислення середньої кількості захворювань за період
run_sql("""
    CREATE OR REPLACE FUNCTION pandemic.cases_per_period(
        p_cases NUMERIC,
        p_divisor NUMERIC
    )
    RETURNS NUMERIC
    LANGUAGE SQL
    IMMUTABLE
    AS $$
        SELECT p_cases / NULLIF(p_divisor, 0);
    $$;

    SELECT
        pandemic.cases_per_period(120, 12) AS monthly_demo,
        pandemic.cases_per_period(120, 4) AS quarterly_demo,
        pandemic.cases_per_period(120, 2) AS half_year_demo,
        pandemic.cases_per_period(120, 0) AS zero_divisor_demo;
""")

OK: -1 rows


,monthly_demo,quarterly_demo,half_year_demo,zero_divisor_demo
0,10.0000000000000000,30.0000000000000000,60.0000000000000000,None


In [23]:
run_sql("""
    SELECT
        dim."Entity",
        fact."Year",
        fact."Number_rabies" AS rabies_per_year,
        ROUND(
            pandemic.cases_per_period(fact."Number_rabies", 12), 2
        ) AS rabies_per_month,
        ROUND(
            pandemic.cases_per_period(fact."Number_rabies", 4), 2
        ) AS rabies_per_quarter,
        ROUND(
            pandemic.cases_per_period(fact."Number_rabies", 2), 2
        ) AS rabies_per_half_year
    FROM pandemic.disease_statistics AS fact
    JOIN pandemic.entities AS dim ON dim.id = fact.entity_id
    WHERE fact."Number_rabies" IS NOT NULL
    ORDER BY dim."Entity", fact."Year"
    LIMIT 20;
""")

,Entity,Year,rabies_per_year,rabies_per_month,rabies_per_quarter,rabies_per_half_year
0,Afghanistan,1990,2.145772,0.18,0.54,1.07
1,Afghanistan,1991,2.3209748,0.19,0.58,1.16
2,Afghanistan,1992,2.6180408,0.22,0.65,1.31
3,Afghanistan,1993,2.8531172,0.24,0.71,1.43
4,Afghanistan,1994,2.9899871,0.25,0.75,1.49
5,Afghanistan,1995,3.126361,0.26,0.78,1.56
6,Afghanistan,1996,3.281437,0.27,0.82,1.64
7,Afghanistan,1997,3.446122,0.29,0.86,1.72
8,Afghanistan,1998,3.5570247,0.30,0.89,1.78
9,Afghanistan,1999,3.607037,0.30,0.90,1.80


In [24]:
# 6. Перевірка результатів
run_sql("""
    SELECT
        COUNT(*) AS fact_rows,
        COUNT("Number_rabies") AS numeric_rabies_rows,
        COUNT(*) - COUNT("Number_rabies") AS null_or_non_numeric_rabies_rows,
        MIN("Year") AS min_year,
        MAX("Year") AS max_year
    FROM pandemic.disease_statistics;

    SELECT
        BOOL_AND(
            pandemic.years_since("Year") =
            EXTRACT(
                YEAR FROM AGE(CURRENT_DATE, MAKE_DATE("Year", 1, 1))
            )::INTEGER
        ) AS year_function_matches_builtin
    FROM pandemic.disease_statistics;

    SELECT
        pandemic.cases_per_period(120, 12) = 10 AS monthly_ok,
        pandemic.cases_per_period(120, 4) = 30 AS quarterly_ok,
        pandemic.cases_per_period(120, 2) = 60 AS half_year_ok,
        pandemic.cases_per_period(120, 0) IS NULL AS zero_division_safe;
""")

,fact_rows,numeric_rabies_rows,null_or_non_numeric_rabies_rows,min_year,max_year
0,10521,6780,3741,1920,2023


,year_function_matches_builtin
0,True


,monthly_ok,quarterly_ok,half_year_ok,zero_division_safe
0,True,True,True,True


# Reflection
1. Створено схему pandemic і завантажено повний набір infectious_cases.csv.
2. Дані нормалізовано до 3НФ у дві таблиці: entities (унікальні Entity / Code) і disease_statistics (рік, усі кількісні показники, зовнішній ключ).
3. Для Number_rabies виконано групування та обчислено середнє, мінімум, максимум і суму для кожної сутності; показано 10 записів із найбільшим середнім.
4. Дату 1 січня і різницю в повних роках отримано через MAKE_DATE, CURRENT_DATE, AGE, EXTRACT.
5. Створено та перевірено дві PostgreSQL-функції: years_since() і cases_per_period() (періоди 12 / 4 / 2).